# Feature Importance Workflow (Steps 3a–3b)

Run this **after** cohorts exist (Step 2; see `1_cohort_workflow.ipynb`). One cell per cohort for Step 3a and for Step 3b; run Configuration and Sync once, then the cohort cell(s) you need.

## Order of operations

1. **Configuration** — Project root, data root, cohort/age-band list (run once).
2. **Sync inputs** — Sync `gold/cohorts` from S3 to local/NVMe (idempotent).
3. **Step 3a** — MC-CV feature importance: one runnable cell per cohort × age_band (e.g. opioid_ed/13-24, opioid_ed/25-44, …). Run the cell for the combination you want; checkpoint skip per cohort/age_band.
4. **Update cohort data before Step 3b** — Sync `gold/cohorts`, `gold/medical`, `gold/pharmacy` from S3 (run once before any Step 3b cell) so 3b uses the latest data.
5. **Step 3b** — Feature Importance EDA (interactive): one runnable cell **per cohort**. Each runs that cohort for all its age_bands. Checkpoint skip per cohort/age_band.

## Cohorts

| Cohort | Age bands |
|--------|-----------|
| **OPIOID_ED** | 13-24, 25-44, 45-54, 55-64 |
| **POLYPHARMACY** (non_opioid_ed) | 65-74, 75-84, 85-94 |

## Reference

- Step 3a: `3a_feature_importance/run_mc_feature_importance.py`
- Step 3b: `3b_feature_importance_eda/feature_importance_eda_workflow.py`; interactive EDA in `3b_feature_importance_eda/`
- Step 6: `6_final_model/` — model training and selection.

## Configuration

In [2]:
import sys
import os
from pathlib import Path
import subprocess
import logging

from IPython.display import Image, display

PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
if not (PROJECT_ROOT / "3a_feature_importance").exists():
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from py_helpers.env_utils import get_data_root
from py_helpers.workflow_sync_checkpoint import (
    sync_s3_to_local,
    check_step_checkpoint_exists,
    save_step_checkpoint,
)
from py_helpers.feature_importance_heatmap import create_aggregated_fi_heatmap

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

PYTHON_BIN = Path(sys.executable)
S3_BUCKET = os.environ.get("PGX_S3_BUCKET", "pgxdatalake")
DATA_ROOT = get_data_root()
AWS_PROFILE = os.environ.get("AWS_PROFILE")

COHORTS = {
    "opioid_ed": ["13-24", "25-44", "45-54", "55-64"],
    "non_opioid_ed": ["65-74", "75-84", "85-94"],
}

# Set True to overwrite/force Step 3a feature importance (rerun even when results exist)
FORCE_FEATURE_IMPORTANCE = False

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root (NVMe/local): {DATA_ROOT}")
print(f"Python: {PYTHON_BIN}")
print(f"Force feature importance (overwrite): {FORCE_FEATURE_IMPORTANCE}")

# Step 3a outputs base (same as run_mc_feature_importance.py)
OUTPUTS_BASE_3A = Path(os.environ.get("PGX_FEATURE_IMPORTANCE_OUTPUTS", str(PROJECT_ROOT / "3a_feature_importance" / "outputs")))

Project root: /home/pgx3874/pgx-analysis
Data root (NVMe/local): /mnt/nvme
Python: /home/pgx3874/jupyter-env/bin/python3.11
Force feature importance (overwrite): False


## Sync required inputs from S3 to NVMe (idempotent)

Sync **gold/cohorts** from S3 so Step 3a can read cohort parquet from local/NVMe. **Idempotent:** `aws s3 sync` only updates changed or missing files.

In [3]:
# Sync gold/cohorts from S3 to local/NVMe (required for 3a feature importance)
s3_cohorts = f"s3://{S3_BUCKET}/gold/cohorts/"
local_cohorts = DATA_ROOT / "gold" / "cohorts"
ok = sync_s3_to_local(s3_cohorts, local_cohorts, profile=AWS_PROFILE)
print(f"  gold/cohorts: {'OK' if ok else 'FAILED or skipped (no AWS CLI)'}")

INFO: Sync completed: s3://pgxdatalake/gold/cohorts/ -> /mnt/nvme/gold/cohorts


  gold/cohorts: OK


## pgx-repository baseline (aggregated feature importance)

Current aggregated feature importance files in **pgx-repository** (used by Step 3a second pass as baseline). Run this cell to confirm row counts and feature counts before running Step 3a.

In [4]:
# Load and display pgx-repository baseline summary (all cohort/age_band)
import sys
sys.path.insert(0, str(PROJECT_ROOT / "3a_feature_importance"))
from load_pgx_repo_fi import get_baseline_summary_df

baseline_df = get_baseline_summary_df()
display(baseline_df)

,cohort,age_band,rows,unique_features,sample
0,opioid_ed,13-24,11058,11058,"item_80307, item_SUBOXONE, item_H0020"
1,opioid_ed,25-44,4962,4962,"item_SUBOXONE, item_H0020, item_80307"
2,opioid_ed,45-54,3534,3534,"item_H0020, item_80307, item_SUBOXONE"
3,opioid_ed,55-64,3882,3882,"item_H0020, item_G0483, item_80307"
4,non_opioid_ed,65-74,1519,1519,"item_HYDROCODONE BITARTRATE/AC, item_PREDNISON..."
5,non_opioid_ed,75-84,1184,1184,"item_FUROSEMIDE, item_PREDNISONE, item_HYDROCO..."
6,non_opioid_ed,85-94,848,848,"item_FUROSEMIDE, item_CEPHALEXIN, item_PREDNISONE"


## Step 3a: MC-CV feature importance

Monte Carlo CV feature importance (CatBoost, XGBoost, XGBoost RF). **One runnable cell per cohort × age_band** so you can run and troubleshoot a single combination. Skipped when checkpoint exists for that cohort/age_band. Set **FORCE_FEATURE_IMPORTANCE = True** in Configuration to overwrite/force rerun (passes **--force** to the script). After running the age_band cells for a cohort, run that cohort’s heatmap cell to build the aggregated feature importance heatmap (feature × age band).

### Cohort 1: OPIOID_ED — one cell per age_band

In [ ]:
# Step 3a: opioid_ed / 13-24 only
step_name_3a = "3a_feature_importance"
script_3a = PROJECT_ROOT / "3a_feature_importance" / "run_mc_feature_importance.py"
cohort, age_band = "opioid_ed", "13-24"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Step 3a: opioid_ed / 25-44 only
cohort, age_band = "opioid_ed", "25-44"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Step 3a: opioid_ed / 45-54 only
cohort, age_band = "opioid_ed", "45-54"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Step 3a: opioid_ed / 55-64 only
cohort, age_band = "opioid_ed", "55-64"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Aggregated feature importance heatmap for OPIOID_ED (run after age_band cells above)
cohort = "opioid_ed"
heatmap_path = create_aggregated_fi_heatmap(cohort, COHORTS[cohort], OUTPUTS_BASE_3A, top_n=50)
if heatmap_path and heatmap_path.exists():
    print(f"Heatmap saved: {heatmap_path}")
    display(Image(filename=str(heatmap_path)))
else:
    print("Heatmap skipped (need at least 2 age bands with aggregated CSVs).")

### Cohort 2: POLYPHARMACY (non_opioid_ed) — one cell per age_band

In [ ]:
# Step 3a: non_opioid_ed / 65-74 only
cohort, age_band = "non_opioid_ed", "65-74"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Step 3a: non_opioid_ed / 75-84 only
cohort, age_band = "non_opioid_ed", "75-84"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Step 3a: non_opioid_ed / 85-94 only
cohort, age_band = "non_opioid_ed", "85-94"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Aggregated feature importance heatmap for POLYPHARMACY (run after age_band cells above)
cohort = "non_opioid_ed"
heatmap_path = create_aggregated_fi_heatmap(cohort, COHORTS[cohort], OUTPUTS_BASE_3A, top_n=50)
if heatmap_path and heatmap_path.exists():
    print(f"Heatmap saved: {heatmap_path}")
    display(Image(filename=str(heatmap_path)))
else:
    print("Heatmap skipped (need at least 2 age bands with aggregated CSVs).")

## Final combined feature importance heatmap (per cohort, all age bands)

One heatmap **per cohort**: feature × age band for that cohort. Rows = top features (union across age bands), columns = age bands for that cohort. Run after the Step 3a age_band cells for each cohort.

In [ ]:
# Combined heatmap per cohort: feature × age band for each cohort (all age bands for that cohort)
for cohort in COHORTS:
    heatmap_path = create_aggregated_fi_heatmap(cohort, COHORTS[cohort], OUTPUTS_BASE_3A, top_n=80)
    if heatmap_path and heatmap_path.exists():
        print(f"Combined heatmap for {cohort} saved: {heatmap_path}")
        display(Image(filename=str(heatmap_path)))
    else:
        print(f"Combined heatmap for {cohort} skipped (need at least 2 age bands with aggregated CSVs).")

## Final: Update model features with target leakage removed

Step 4 uses `cohort_feature_importance.csv` to filter **case events** when building model data; it does **not** remove target leakage from the feature list. Target leakage is removed in Step 3b (Filter and Refine). This cell explicitly ensures each final `cohort_feature_importance.csv` has all BupaR-identified post-target leakage features removed, so the list passed to Step 4 is clean. Run after Step 3b cohort cells.

## Update cohort data before Step 3b

Sync **gold/cohorts**, **gold/medical**, and **gold/pharmacy** from S3 so Step 3b has the latest data when building model_events (gold cohort filtered by 3a FI + admin removed). Run this cell **once before running any Step 3b cell** to keep the pipeline seamless between 3a and 3b.

In [ ]:
# Sync cohort and gold medical/pharmacy before Step 3b (idempotent)
S3_BUCKET = os.environ.get("PGX_S3_BUCKET", "pgxdatalake")
data_root = get_data_root()
syncs = [
    (f"s3://{S3_BUCKET}/gold/cohorts/", data_root / "gold" / "cohorts"),
    (f"s3://{S3_BUCKET}/gold/medical/", data_root / "gold" / "medical"),
    (f"s3://{S3_BUCKET}/gold/pharmacy/", data_root / "gold" / "pharmacy"),
]
for s3_prefix, local_dir in syncs:
    ok = sync_s3_to_local(s3_prefix, local_dir, profile=AWS_PROFILE)
    print(f"  {local_dir.name}: {'OK' if ok else 'FAILED or skipped'}")
print("Cohort data updated. Ready for Step 3b.")

## Step 3b: Feature Importance EDA (interactive)

BupaR post-target analysis and refined `cohort_feature_importance.csv` in `3b_feature_importance_eda/outputs/`. **One notebook cell per cohort** so you can run interactively: run the cell for the cohort you want. Each cell runs that cohort for all its age_bands; skipped when checkpoint exists for that cohort/age_band.

### Cohort 1: OPIOID_ED

In [ ]:
# Step 3b for OPIOID_ED (all age_bands); skip if checkpoint exists per cohort/age_band
STEP3B_DIR = PROJECT_ROOT / "3b_feature_importance_eda"
script_3b = STEP3B_DIR / "feature_importance_eda_workflow.py"
step_name_3b = "3b_feature_importance_eda"
cohort = "opioid_ed"
for age_band in COHORTS[cohort]:
    if check_step_checkpoint_exists(step_name_3b, cohort, age_band, logger):
        print(f"Step 3b already completed for {cohort}/{age_band}. Skipping.")
        continue
    print(f"Running Step 3b for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3b), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3b, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")
print(f"Step 3b outputs: {STEP3B_DIR / 'outputs'}")

Running Step 3b for opioid_ed/13-24...
🖥️  Detected OS: Linux
   Using Linux/EC2 path
   Using Linux/EC2 Python: /home/pgx3874/jupyter-env/bin/python3.11
   Using Linux/EC2 Rscript: /usr/local/bin/Rscript
✅ OS detection and path setup complete

📋 Configuration:
   Cohort: opioid_ed
   Age Band: 13-24 (13_24)
   Output Directory: /home/pgx3874/pgx-analysis/3b_feature_importance_eda/outputs/opioid_ed/13_24

💡 Tip: Set cohort/age_band via:
   - Command-line: python feature_importance_eda_workflow.py --cohort opioid_ed --age-band 13-24
   - Environment: export FEATURE_IMPORTANCE_EDA_COHORT=opioid_ed && export FEATURE_IMPORTANCE_EDA_AGE_BAND=13-24
   - Manual: Edit COHORT and AGE_BAND variables above

✅ Configuration loaded
   Project Root: /home/pgx3874/pgx-analysis
   Cohort: opioid_ed
   Age Band: 13-24 (13_24)
   Python Binary: /home/pgx3874/jupyter-env/bin/python3.11
   Rscript Binary: /usr/local/bin/Rscript
   Output Directory: /home/pgx3874/pgx-analysis/3b_feature_importance_eda/outp

INFO: ✓ Saved checkpoint to s3://pgx-repository/pipeline_checkpoints/3b_feature_importance_eda/opioid_ed/13_24/checkpoint.json


  opioid_ed/13-24 exit code: 0
Running Step 3b for opioid_ed/25-44...


### Cohort 2: POLYPHARMACY (non_opioid_ed)

In [ ]:
# Step 3b for POLYPHARMACY (all age_bands); skip if checkpoint exists per cohort/age_band
cohort = "non_opioid_ed"
for age_band in COHORTS[cohort]:
    if check_step_checkpoint_exists(step_name_3b, cohort, age_band, logger):
        print(f"Step 3b already completed for {cohort}/{age_band}. Skipping.")
        continue
    print(f"Running Step 3b for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3b), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3b, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

## Update Model Data

In [ ]:
# Update final model features: remove BupaR-identified target leakage from cohort_feature_importance
import pandas as pd

OUTPUTS_3B = PROJECT_ROOT / "3b_feature_importance_eda" / "outputs"
updated_count = 0
for cohort in COHORTS:
    for age_band in COHORTS[cohort]:
        age_fname = age_band.replace("-", "_")
        refined_path = OUTPUTS_3B / cohort / age_fname / f"{cohort}_{age_fname}_cohort_feature_importance.csv"
        bupar_path = OUTPUTS_3B / cohort / age_fname / f"{cohort}_{age_fname}_bupar_post_target_analysis.csv"
        if not refined_path.exists():
            print(f"  Skip {cohort}/{age_band}: no cohort_feature_importance.csv")
            continue
        if not bupar_path.exists():
            print(f"  Skip {cohort}/{age_band}: no BupaR post-target analysis CSV")
            continue
        refined = pd.read_csv(refined_path)
        bupar = pd.read_csv(bupar_path)
        if "feature" not in refined.columns:
            print(f"  Skip {cohort}/{age_band}: no 'feature' column in refined CSV")
            continue
        leakage = set()
        if "is_post_target_leakage" in bupar.columns and "feature" in bupar.columns:
            leakage = set(bupar.loc[bupar["is_post_target_leakage"] == 1, "feature"].dropna().astype(str).tolist())
        if not leakage:
            print(f"  {cohort}/{age_band}: no leakage features to remove")
            continue
        n_before = len(refined)
        refined_clean = refined[~refined["feature"].astype(str).isin(leakage)].copy()
        n_after = len(refined_clean)
        removed = n_before - n_after
        if removed > 0:
            refined_clean.to_csv(refined_path, index=False)
            print(f"  {cohort}/{age_band}: removed {removed} leakage features; {n_after} features saved to {refined_path.name}")
            updated_count += 1
        else:
            print(f"  {cohort}/{age_band}: leakage set had no overlap with refined features (already clean)")
if updated_count > 0:
    print(f"\nUpdated {updated_count} cohort_feature_importance file(s). Ready for Step 4.")
else:
    print("\nNo files updated (already clean or missing BupaR/refined outputs).")